In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a program that performs a 1D convolution operation. Given an input array and a kernel (filter), compute the convolved
  output. The convolution should be performed with a "valid" boundary condition, meaning the kernel is only applied
  where it fully overlaps with the input.
</p>

<svg width="420" height="210" viewBox="0 0 420 210" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;" font-family="monospace" font-size="13">
  <!-- Background -->
  <rect width="420" height="210" rx="8" fill="#222"/>

  <!-- "input" label -->
  <text x="16" y="38" fill="#999" font-size="11">input</text>

  <!-- Input cells -->
  <rect x="65"  y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="120" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="175" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="230" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <rect x="285" y="20" width="50" height="32" rx="3" fill="#333" stroke="#555" stroke-width="1"/>
  <!-- Input values -->
  <text x="90"  y="41" text-anchor="middle" fill="#ccc">1</text>
  <text x="145" y="41" text-anchor="middle" fill="#ccc">2</text>
  <text x="200" y="41" text-anchor="middle" fill="#ccc">3</text>
  <text x="255" y="41" text-anchor="middle" fill="#ccc">4</text>
  <text x="310" y="41" text-anchor="middle" fill="#ccc">5</text>

  <!-- Kernel highlight window over first 3 input cells -->
  <rect x="63" y="18" width="164" height="36" rx="4" fill="none" stroke="#4477bb" stroke-width="2" stroke-dasharray="5,3"/>

  <!-- "kernel" label -->
  <text x="16" y="86" fill="#999" font-size="11">kernel</text>

  <!-- Kernel cells (aligned under first 3 input cells) -->
  <rect x="65"  y="68" width="50" height="32" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <rect x="120" y="68" width="50" height="32" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <rect x="175" y="68" width="50" height="32" rx="3" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <!-- Kernel values -->
  <text x="90"  y="89" text-anchor="middle" fill="#88bbff">1</text>
  <text x="145" y="89" text-anchor="middle" fill="#88bbff">0</text>
  <text x="200" y="89" text-anchor="middle" fill="#88bbff">-1</text>

  <!-- Multiplication signs between pairs -->
  <text x="90"  y="118" text-anchor="middle" fill="#777" font-size="11">1&#xd7;1</text>
  <text x="145" y="118" text-anchor="middle" fill="#777" font-size="11">2&#xd7;0</text>
  <text x="200" y="118" text-anchor="middle" fill="#777" font-size="11">3&#xd7;(-1)</text>

  <!-- Computation line -->
  <text x="145" y="140" text-anchor="middle" fill="#aaa" font-size="12">= 1 + 0 + (-3) = -2</text>

  <!-- Arrow down to output -->
  <line x1="145" y1="148" x2="145" y2="168" stroke="#4477bb" stroke-width="1.5" marker-end="url(#arrowhead)"/>
  <defs>
    <marker id="arrowhead" markerWidth="8" markerHeight="6" refX="8" refY="3" orient="auto">
      <polygon points="0 0, 8 3, 0 6" fill="#4477bb"/>
    </marker>
  </defs>

  <!-- "output" label -->
  <text x="16" y="187" fill="#999" font-size="11">output</text>

  <!-- Output cell -->
  <rect x="120" y="170" width="50" height="30" rx="3" fill="#1a3a1a" stroke="#44aa44" stroke-width="1.5"/>
  <text x="145" y="190" text-anchor="middle" fill="#66dd66" font-weight="bold">-2</text>

  <!-- Ellipsis for remaining output -->
  <text x="195" y="190" fill="#666" font-size="14">&#x2026;</text>
</svg>

<p>
  The input consists of two arrays:
<ul>
  <li><code>input</code>: A 1D array of 32-bit floating-point numbers.</li>
  <li><code>kernel</code>: A 1D array of 32-bit floating-point numbers representing the convolution kernel.</li>
</ul>
The output should be written to the <code>output</code> array, which will have a size of <code>input_size - kernel_size + 1</code>.
</p>

<p>
  The convolution operation is defined mathematically as:
</p>

$$
output[i] = \sum_{j=0}^{kernel\_size-1} input[i + j] \cdot kernel[j]
$$

<p>
  where $i$ ranges from 0 to $input\_size - kernel\_size$.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The
    <code>solve</code> function signature must remain unchanged
  </li>
  <li>The final result must be stored in the array
    <code>output</code>
  </li>
</ul>

<h2>Example 1:</h2>
<pre>
Input: input = [1, 2, 3, 4, 5], kernel = [1, 0, -1]
Output: [-2, -2, -2]
</pre>

<h2>Example 2:</h2>
<pre>
Input: input = [2, 4, 6, 8], kernel = [0.5, 0.2]
Output: [1.8, 3.2, 4.6]
</pre>

<h2>Constraints</h2>

<ul>
  <li>1 &le; <code>input_size</code> &le; 1,500,000</li>
  <li>1 &le; <code>kernel_size</code> &le; 2047</li>
  <li><code>kernel_size</code> &le; <code>input_size</code></li>

  <li>Performance is measured with <code>input_size</code> = 1,500,000, <code>kernel_size</code> = 2,047</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void convolution_1d_kernel(const float* input, const float* kernel, float* output,
                                      int input_size, int kernel_size) {}

// input, kernel, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* input, const float* kernel, float* output, int input_size,
                      int kernel_size) {
    int output_size = input_size - kernel_size + 1;
    int threadsPerBlock = 256;
    int blocksPerGrid = (output_size + threadsPerBlock - 1) / threadsPerBlock;

    convolution_1d_kernel<<<blocksPerGrid, threadsPerBlock>>>(input, kernel, output, input_size,
                                                              kernel_size);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, kernel, output are tensors on the GPU
@cute.jit
def solve(
    input: cute.Tensor,
    kernel: cute.Tensor,
    output: cute.Tensor,
    input_size: cute.Int32,
    kernel_size: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input, kernel are tensors on the GPU
@jax.jit
def solve(input: jax.Array, kernel: jax.Array, input_size: int, kernel_size: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def convolution_1d_kernel(
    input: UnsafePointer[Float32, MutExternalOrigin],
    kernel: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    input_size: Int32,
    kernel_size: Int32,
):
    pass


# input, kernel, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    kernel: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    input_size: Int32,
    kernel_size: Int32,
) raises:
    var output_size = input_size - kernel_size + 1
    var threadsPerBlock: Int32 = 256
    var ctx = DeviceContext()

    var blocksPerGrid = ceildiv(output_size, threadsPerBlock)

    var _kernel = ctx.compile_function[convolution_1d_kernel, convolution_1d_kernel]()
    ctx.enqueue_function(
        _kernel,
        input,
        kernel,
        output,
        input_size,
        kernel_size,
        grid_dim=blocksPerGrid,
        block_dim=threadsPerBlock,
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, kernel, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_size: int,
    kernel_size: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


@triton.jit
def conv1d_kernel(input, kernel, output, input_size, kernel_size, BLOCK_SIZE: tl.constexpr):
    pass


# input, kernel, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_size: int,
    kernel_size: int,
):
    BLOCK_SIZE = 1024
    n_blocks = triton.cdiv(input_size - kernel_size + 1, BLOCK_SIZE)
    grid = (n_blocks,)

    conv1d_kernel[grid](input, kernel, output, input_size, kernel_size, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="1D Convolution", atol=1e-04, rtol=1e-04, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        input: torch.Tensor,
        kernel: torch.Tensor,
        output: torch.Tensor,
        input_size: int,
        kernel_size: int,
    ):
        assert input.shape == (input_size,)
        assert kernel.shape == (kernel_size,)
        assert output.shape == (input_size - kernel_size + 1,)
        assert input.dtype == kernel.dtype == output.dtype
        assert input.device == kernel.device == output.device

        # Create strided view of input for all windows
        windows = input.unfold(0, kernel_size, 1)

        # Use einsum for explicit cross-correlation
        # 'ij,j->i' means: for each window i, multiply with kernel j and sum over j
        output.copy_(torch.einsum("ij,j->i", windows, kernel))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_float), "in"),
            "kernel": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "input_size": (ctypes.c_int, "in"),
            "kernel_size": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        input_tensor = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0], device="cuda", dtype=dtype)
        kernel_tensor = torch.tensor([1.0, 0.0, -1.0], device="cuda", dtype=dtype)
        output_tensor = torch.empty(3, device="cuda", dtype=dtype)
        return {
            "input": input_tensor,
            "kernel": kernel_tensor,
            "output": output_tensor,
            "input_size": 5,
            "kernel_size": 3,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        test_specs = [
            # Basic test cases
            ("basic_5x3", [1.0, 2.0, 3.0, 4.0, 5.0], [1.0, 0.0, -1.0]),
            ("basic_4x2", [2.0, 4.0, 6.0, 8.0], [0.5, 0.2]),
            ("identity_kernel", [1.0, 2.0, 3.0, 4.0], [1.0]),
            ("edge_detection", [1.0, 1.0, 1.0, 0.0, 0.0, 0.0], [1.0, -1.0]),
            ("smoothing", [1.0, 2.0, 3.0, 4.0, 5.0], [0.25, 0.5, 0.25]),
        ]

        test_cases = []
        for _, input_vals, kernel_vals in test_specs:
            input_size = len(input_vals)
            kernel_size = len(kernel_vals)
            output_size = input_size - kernel_size + 1
            test_cases.append(
                {
                    "input": torch.tensor(input_vals, device="cuda", dtype=dtype),
                    "kernel": torch.tensor(kernel_vals, device="cuda", dtype=dtype),
                    "output": torch.empty(output_size, device="cuda", dtype=dtype),
                    "input_size": input_size,
                    "kernel_size": kernel_size,
                }
            )

        # Random test cases with different sizes
        for _, input_size, kernel_size in [
            ("small_conv", 10, 3),
            ("medium_conv", 100, 7),
            ("large_conv", 1000, 15),
            ("wide_kernel", 50, 20),
            ("narrow_kernel", 200, 2),
        ]:
            output_size = input_size - kernel_size + 1
            test_cases.append(
                {
                    "input": torch.empty(input_size, device="cuda", dtype=dtype).uniform_(
                        -10.0, 10.0
                    ),
                    "kernel": torch.empty(kernel_size, device="cuda", dtype=dtype).uniform_(
                        -1.0, 1.0
                    ),
                    "output": torch.empty(output_size, device="cuda", dtype=dtype),
                    "input_size": input_size,
                    "kernel_size": kernel_size,
                }
            )

        # Edge cases
        for _, input_size, kernel_size in [
            ("min_input", 1, 1),
            ("kernel_equals_input", 10, 10),
            ("large_input_small_kernel", 10000, 3),
        ]:
            output_size = input_size - kernel_size + 1
            test_cases.append(
                {
                    "input": torch.empty(input_size, device="cuda", dtype=dtype).uniform_(
                        -1.0, 1.0
                    ),
                    "kernel": torch.empty(kernel_size, device="cuda", dtype=dtype).uniform_(
                        -0.1, 0.1
                    ),
                    "output": torch.empty(output_size, device="cuda", dtype=dtype),
                    "input_size": input_size,
                    "kernel_size": kernel_size,
                }
            )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        input_size, kernel_size = 1500000, 2047  # Large convolution for performance testing
        output_size = input_size - kernel_size + 1
        return {
            "input": torch.empty(input_size, device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
            "kernel": torch.empty(kernel_size, device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
            "output": torch.empty(output_size, device="cuda", dtype=dtype),
            "input_size": input_size,
            "kernel_size": kernel_size,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
